# Seminar 3 — Missions 3 and 4

Follow one comment from text to tokens to Yes/No logits, then score the class dev split and understand its AUC.

**Run on Modal:** on the class server, run `modal run notebook.py` from `~/course/seminar_3`. Open the printed HTTPS link on your laptop. See [the launch guide](https://github.com/hakob-petro/aca_moderation_practicum/blob/main/seminar_3/NOTEBOOK.md).

Run cells in order with **Shift+Enter**. Three cells marked **Your turn** need your input; running all cells before filling them will stop at the first exercise. You need to write two score lines, five comments, and one AUC answer.

The GPU is billed while this session is open, including idle time. The default limit is 90 minutes. Save with **Ctrl+S** (or **Cmd+S**) and finish the final export cell before stopping the launcher with **Ctrl+C**. Closing this tab alone does not stop the GPU.


## Set up the session

The browser is on your laptop; this Python kernel and its GPU are on Modal. `~/s3` below is on Modal until you download the export back to the class server. Your notebook and `s3` files live in the launcher's persistent workspace.


In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import subprocess
import sys
import time

COURSE = Path('/opt/course/seminar_3')
assert COURSE.exists(), 'Open this notebook using modal run notebook.py; see NOTEBOOK.md.'
sys.path.insert(0, str(COURSE))
import scorer
import prepare_data
from auc import auc, column_averaged_auc

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), 'This lesson needs the GPU supplied by the Modal launcher.'
S3 = Path.home() / 's3'
S3.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Files:', S3.resolve())
print('Model:', scorer.MODEL)
print('Pinned revision:', scorer.REVISION)


## Mission 3 — Text becomes tokens

A tokenizer converts text into token IDs. Tokens can be words, word pieces, spaces or punctuation. A leading space can change a token's ID. In a notebook string, `$5` stays literal; a shell double-quoted argument can interpret `$5` as a variable.


In [ ]:
tok = AutoTokenizer.from_pretrained(scorer.MODEL, revision=scorer.REVISION)
tok.padding_side = 'left'
sentence = 'Get 1000 Instagram followers for $5'
print(tok.tokenize(sentence))
print(tok.encode(sentence, add_special_tokens=False))
tokens_in_sentence = len(tok.encode(sentence, add_special_tokens=False))
print('Token count:', tokens_in_sentence)

[yes_id] = tok.encode('Yes', add_special_tokens=False)
[space_yes_id] = tok.encode(' Yes', add_special_tokens=False)
[no_id] = tok.encode('No', add_special_tokens=False)
print({'Yes': yes_id, ' Yes': space_yes_id, 'No': no_id})


## The conversation becomes one prompt

We use the same rule and examples as the command-line scorer. Compare the endings with thinking off and on. Here we need the next token to be the answer, so thinking is off. This experiment does not measure performance after a generated reasoning sequence.


In [ ]:
chat = scorer.chat_for('legal', 'Can my landlord keep my deposit?')
print(tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False))
print('\nWith thinking enabled, the ending is:')
print(repr(tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=True)[-100:]))


## One pass through the model

Qwen3-0.6B is already pretrained and post-trained, but has not had our task-specific training. We load it once and reuse it in both missions. T4 uses float16; newer GPUs use bfloat16. Both store each weight in two bytes, although inference needs additional memory for activations and other work.

`logits.shape` is `(batch, prompt tokens, vocabulary size)`. The last prompt position contains scores for the next token.


In [ ]:
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
model = AutoModelForCausalLM.from_pretrained(scorer.MODEL, revision=scorer.REVISION, dtype=dtype).to('cuda')
model.eval()
inputs = tok.apply_chat_template(
    chat, add_generation_prompt=True, enable_thinking=False,
    return_tensors='pt', return_dict=True,
).to('cuda')
with torch.inference_mode():
    logits = model(**inputs).logits
logits_shape = tuple(logits.shape)
print('dtype:', dtype)
print('logits.shape:', logits_shape)
last = logits[0, -1].float()


## Inspect the next token

Softmax gives probabilities across the **whole vocabulary**. The five most likely tokens need not all be Yes or No. We will explicitly compare those two candidates, just as likelihood-based MMLU compares its prescribed A/B/C/D candidates.


In [ ]:
probs = torch.softmax(last, dim=-1)
top = torch.topk(probs, 5)
for probability, token_id in zip(top.values, top.indices):
    print(f'{probability.item():.4f}  {tok.decode(token_id.item())!r}')
print('Logit Yes:', last[yes_id].item())
print('Logit No: ', last[no_id].item())
print('Full-vocabulary mass on Yes + No:', (probs[yes_id] + probs[no_id]).item())


## Your turn — The score and conditional probability

Replace the two `None` values. The score is `logit(Yes) - logit(No)`; `torch.sigmoid(score)` maps it to 0–1. Use `last`, `yes_id` and `no_id`.

This probability is conditional on choosing Yes or No. It is not automatically calibrated as the probability that a comment violates the rule. Comparing logits, testing `score > 0`, and testing this conditional probability `> 0.5` produce the same decision.


In [ ]:
# Replace None with your two expressions.
logit_difference = None
conditional_p_yes = None
assert logit_difference is not None and conditional_p_yes is not None, 'Fill in both expressions above, then rerun this cell.'
assert torch.isclose(torch.as_tensor(logit_difference), last[yes_id] - last[no_id]), 'Check the subtraction order.'
assert torch.isclose(torch.as_tensor(conditional_p_yes), torch.sigmoid(last[yes_id] - last[no_id])), 'Apply sigmoid to the difference.'
landlord_p_yes = float(conditional_p_yes)
print(f'Score: {float(logit_difference):+.4f}; conditional P(Yes): {landlord_p_yes:.4f}')
print('Selected answer:', 'Yes' if float(logit_difference) > 0 else 'No')


In [ ]:
# Write the observed values in the format the existing mission check expects.
answers_m3 = {
    'tokens_in_sentence': tokens_in_sentence,
    'yes_id': yes_id,
    'space_yes_id': space_yes_id,
    'no_id': no_id,
    'logits_shape': logits_shape,
    'p_yes_landlord': f'{landlord_p_yes:.2f}',
}
(S3 / 'm3.txt').write_text(''.join(f'{key} = {value}\n' for key, value in answers_m3.items()))
print((S3 / 'm3.txt').read_text())


## Reuse the loaded model

This function repeats the same next-token calculation for a batch. Left padding puts the end of every prompt at the same position. `logits_to_keep=1` avoids returning vocabulary scores at every earlier position.

Read the final subtraction: it is the operation you just wrote. We keep the raw difference so rounding a probability cannot destroy the score ordering.


In [ ]:
def prompt_for(chat):
    return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def score_chats(chats, batch_size=16):
    texts = [prompt_for(c) for c in chats]
    scores = []
    for start in range(0, len(texts), batch_size):
        batch = tok(texts[start:start + batch_size], return_tensors='pt', padding=True).to('cuda')
        with torch.inference_mode():
            next_logits = model(**batch, logits_to_keep=1).logits[:, -1, :].float()
        scores.extend((next_logits[:, yes_id] - next_logits[:, no_id]).tolist())
    return scores

[again] = score_chats([chat])
print('Same comment through the batch scorer:', scorer.p_yes(again))
# Different batch shapes and precision can cause small numerical changes.
assert abs(scorer.p_yes(again) - landlord_p_yes) < 0.03


## Your turn — Write five comments

Replace the two examples with **at least five of your own**, covering both advertising and legal advice. Try genuine recommendations, promotional links, accounts of a court case and requests for legal advice. Keep each comment on one line.

The two examples below are also in the template and do not count toward the mission check.


In [ ]:
my_comments = [
    ('advertising', 'Get 1000 Instagram followers for $5, DM me or visit cheapfollows.example'),
    ('legal', 'Can my landlord keep my deposit if I leave a month early?'),
]
assert len(my_comments) >= 5, 'Write at least five comments of your own.'
assert {rule for rule, text in my_comments} >= {'advertising', 'legal'}, 'Try both rules.'
assert all(rule in scorer.RULES and text.strip() and '\n' not in text and '\t' not in text for rule, text in my_comments)
(S3 / 'my_comments.txt').write_text(''.join(f'{rule} | {text}\n' for rule, text in my_comments))
scorer.run(score_chats, prompt_for, f'Modal notebook ({torch.cuda.get_device_name(0)})', file=str(S3 / 'my_comments.txt'))


In [ ]:
# Run the same check as the terminal-based mission.
subprocess.run(['bash', str(COURSE / 'check'), '3'], check=True)


**Discuss before moving on:** Why compare candidate scores rather than parse generated text? What changes if we let Qwen generate thinking tokens first? Which of your comments exposed a mismatch between your judgment and the model's answer?

## Mission 4 — AUC by hand

Use the original mission's five-comment exercise:

| Comment | Score | Label |
|---|---:|---|
| A | 0.9 | Violation |
| B | 0.6 | Clean |
| C | 0.4 | Violation |
| D | 0.3 | Clean |
| E | 0.1 | Clean |

Compare each violation with every clean comment. Count a win when the violation scores higher; ties get half credit. Divide by the total number of pairs. Enter your answer before running the verification cell.


In [ ]:
auc_by_hand = None  # Replace None with your fraction or decimal.
assert auc_by_hand is not None, 'Count the pairs in the table and enter your AUC.'
(S3 / 'm4.txt').write_text(f'auc_by_hand = {auc_by_hand}\n')


In [ ]:
toy_labels = [1, 0, 1, 0, 0]
toy_scores = [0.9, 0.6, 0.4, 0.3, 0.1]
positive = [s for y, s in zip(toy_labels, toy_scores) if y == 1]
negative = [s for y, s in zip(toy_labels, toy_scores) if y == 0]
wins = sum(p > n for p in positive for n in negative)
ties = sum(p == n for p in positive for n in negative)
pairs = len(positive) * len(negative)
print(f'({wins} wins + 0.5 × {ties} ties) / {pairs} pairs = {(wins + ties / 2) / pairs:.6f}')
assert abs(float(auc_by_hand) - auc(toy_labels, toy_scores)) < 0.01, 'Recount the pairs.'


## The same AUC as an area

Lower the threshold from above every score to below every score. Passing a violation increases TPR (up); passing a clean comment increases FPR (right). Each clean comment adds width `1/N` at height `violations already passed / P`. Summing those rectangle areas counts correctly ordered violation–clean pairs divided by `P × N`. A tied group uses a diagonal segment and gives tied pairs half credit.

AUC summarizes this whole curve. It does not select a deployment threshold and is not the fraction of comments classified correctly.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(toy_labels, toy_scores, drop_intermediate=False)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(fpr, tpr, marker='o', label=f'AUC = {auc(toy_labels, toy_scores):.3f}')
ax.fill_between(fpr, tpr, alpha=0.15)
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set(xlabel='False positive rate', ylabel='True positive rate', title='Five-comment ROC curve', xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.show()
assert math.isclose(roc_auc_score(toy_labels, toy_scores), auc(toy_labels, toy_scores))


## Prepare the class split and score it

Recreate the same deterministic split as Mission 2: 1,628 train rows and 401 dev rows. This notebook does not fine-tune the model. The labeled examples remain part of the prompt, as in `./score --dev`.

The split keeps identical target comments together, but some dev targets occur inside other rows' prompt examples. We inspect that overlap below; a target-only split does not establish a clean evaluation after fine-tuning.


In [ ]:
prepare_data.main()
train_rows = [json.loads(line) for line in (S3 / 'data/train.jsonl').read_text().splitlines()]
dev_rows = [json.loads(line) for line in (S3 / 'data/dev.jsonl').read_text().splitlines()]
assert len(train_rows) == 1628 and len(dev_rows) == 401
scorer.run(score_chats, prompt_for, f'Modal notebook ({torch.cuda.get_device_name(0)})', dev=True)


In [ ]:
baseline = json.loads((S3 / 'baseline.json').read_text())
with (S3 / 'baseline_predictions.csv').open() as f:
    predictions = list(csv.DictReader(f))
for row in predictions:
    row['row_id'] = int(row['row_id'])
    row['label'] = int(row['label'])
    row['score'] = float(row['score'])
print(json.dumps(baseline, indent=2))
print('First three prediction rows:', predictions[:3])


## Accuracy and AUC answer different questions

Accuracy below uses `score > 0`, the same decision as `logit(Yes) > logit(No)`. Full-score AUC measures ranking. Binary AUC first discards the score magnitude by converting every prediction to 0 or 1. AUC is calculated separately for each rule and then averaged; accuracy is pooled across rows.

A larger full-score AUC is not guaranteed: the small untuned model can rank near chance. Keep the measured result, even if it contradicts your expectation.


In [ ]:
mean_auc, per_rule = column_averaged_auc(predictions)
binary = [{**row, 'score': float(row['score'] > 0)} for row in predictions]
binary_auc, _ = column_averaged_auc(binary)
accuracy = sum((row['score'] > 0) == row['label'] for row in predictions) / len(predictions)
yes_rate = sum(row['score'] > 0 for row in predictions) / len(predictions)
print(f'Full-score AUC: {mean_auc:.4f}; binary AUC: {binary_auc:.4f}')
print(f'Accuracy at zero: {accuracy:.1%}; predicts Yes: {yes_rate:.1%}')
print('Per-rule AUC:', per_rule)


## Move the decision threshold

Change `threshold` and rerun this cell. The marked operating points move; the ROC curves and their AUCs stay the same. TPR divides by actual violations; FPR divides by actual clean comments.

This is exploration on dev, not a test of a threshold chosen independently. For deployment, choose on calibration data and evaluate on separate data, repeating after fine-tuning.


In [ ]:
threshold = 0.0  # Try -1.0, 0.0, 1.0 and 2.0.
fig, ax = plt.subplots(figsize=(7, 4))
for rule in sorted(per_rule):
    rows = [r for r in predictions if r['rule'] == rule]
    labels = [r['label'] for r in rows]
    scores = [r['score'] for r in rows]
    fpr, tpr, _ = roc_curve(labels, scores, drop_intermediate=False)
    [line] = ax.plot(fpr, tpr, label=f'{rule}: AUC {auc(labels, scores):.3f}')
    positives = sum(labels)
    negatives = len(labels) - positives
    tp = sum(r['label'] == 1 and r['score'] > threshold for r in rows)
    fp = sum(r['label'] == 0 and r['score'] > threshold for r in rows)
    ax.scatter([fp / negatives], [tp / positives], color=line.get_color(), marker='x', s=100)
    print(f'{rule}: TPR {tp / positives:.1%}, FPR {fp / negatives:.1%} at score > {threshold}')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
ax.set(xlabel='False positive rate', ylabel='True positive rate', title='Dev ROC curves and selected threshold', xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.show()


## Inspect disagreements with the labels

Look at labeled-clean comments that receive the highest scores. The dataset contains real spam and abusive material. Decide whether the model, the label, or missing context explains each disagreement; a disagreement alone does not prove a label is wrong.


In [ ]:
dev_by_id = {row['row_id']: row for row in dev_rows}
high_clean = sorted((r for r in predictions if r['label'] == 0), key=lambda r: r['score'], reverse=True)[:5]
for row in high_clean:
    print(f"row {row['row_id']} | {row['rule']} | score {row['score']:+.3f}")
    print(dev_by_id[row['row_id']]['body'][:400], '\n')


In [ ]:
# Change this row ID to inspect another example, including its prompt context.
row_id = high_clean[0]['row_id']
row = dev_by_id[row_id]
print('Label:', row['rule_violation'])
print(prompt_for(scorer.build_chat(row['rule'], row['body'], row, row['subreddit'])))


## Check the prompt example overlap

Today this is a pretrained baseline. In Session 4, labeled examples inside training prompts are also information the fine-tuned model can see. Excluding those dev targets is a useful sensitivity check, not a replacement for designing a fresh test split. Exact-text matching does not detect paraphrases.


In [ ]:
example_keys = ['positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']
train_examples = {r[key] for r in train_rows for key in example_keys}
train_targets = {r['body'] for r in train_rows}
assert not train_targets.intersection(r['body'] for r in dev_rows)
exposed = {r['row_id'] for r in dev_rows if r['body'] in train_examples}
self_examples = {r['row_id'] for r in dev_rows if r['body'] in {r[key] for key in example_keys}}
subset = [r for r in predictions if r['row_id'] not in exposed | self_examples]
print(f'{len(exposed)} dev targets occur as training-prompt examples.')
print(f'{len(self_examples)} occur in their own dev prompt examples.')
print(f'{len(subset)} dev rows remain after excluding those matches.')
print('Subset AUC:', column_averaged_auc(subset))


In [ ]:
subprocess.run(['bash', str(COURSE / 'check'), '4'], check=True)


## Export your mission files

Run this cell after both checks pass. It verifies your files again, packages the eight files needed for the terminal checks and Session 4, and commits the export to your persistent Modal volume. It does not download files to the class server yet.

Then save the notebook with **Ctrl+S / Cmd+S**, return to the launch terminal and press **Ctrl+C**. From the same class-server account and Modal profile, run:

```bash
modal run notebook.py --download
./check 3
./check 4
```

The downloader restores files into `~/s3` on the class server and backs up different existing versions. If you launched with `--student NAME`, also pass `--student NAME` to the download command.


In [ ]:
import zipfile
import modal

for mission in ('3', '4'):
    subprocess.run(['bash', str(COURSE / 'check'), mission], check=True)
export_files = ['m3.txt', 'm4.txt', 'my_comments.txt', 'my_scores.tsv', 'baseline.json',
                'baseline_predictions.csv', 'data/train.jsonl', 'data/dev.jsonl']
export_path = Path('/root/work/notebook-results.zip')
with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in export_files:
        archive.write(S3 / filename, arcname=filename)
modal.Volume.from_name(os.environ['ACA_NOTEBOOK_VOLUME']).commit()
print('Export saved:', export_path.name)
print('Save this notebook, then stop the launcher with Ctrl+C and run the download command.')


## Before you finish

Explain why `Yes > No` and `score > 0` agree, what AUC measures, and why AUC does not select a threshold. Write your baseline AUC on the board. Read [Evaluating our moderation model](https://github.com/hakob-petro/aca_moderation_practicum/blob/main/materials/seminar_3/yes_no_scores_and_auc.md) for the full derivation, fine-tuning comparison and evaluation limitations.

To return later, run the same launcher with the same profile and student name. Saved cells and outputs remain; Python variables do not survive a stopped kernel, so rerun the setup and preceding cells. The seed notebook is copied only on your first launch; existing work is never replaced automatically.
